# SIMD 核函数

## 前置要求

学习本节前，建议你已经了解以下内容：

- Ascend C 算子的 Host 侧与 Device 侧分工；
- C/C++ 函数声明、指针参数和模板函数的基本语法。

## 学习目标

完成本节学习后，你应该能够：

- 理解核函数（Kernel Function）在 Ascend C 算子中的作用；
- 理解核函数中 `__global__`和 `__aicore__` 的含义；
- 正确写出 SIMD 核函数的基本定义形式, 掌握模板核函数写法，掌握通过 `<<<...>>>`的形式调用核函数。
- 掌握使用 BuiltIn 关键字如：`__vector__`、`__cube__`、`__mix__(cube, vector)`，设置 Kernel 类型；
- 理解多核并行执行语义，并知道如何通过内置变量或AscendC接口区分不同核处理的数据分片；


## 1. 环境准备

正式开始学习之前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，同时完成了代码目录的创建。保证能正常导入代码以及使用bisheng编译器，完成算子的开发及编译。


In [ ]:
import os
import subprocess
from pathlib import Path

Path("Sources").mkdir(exist_ok=True)

set_env = os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/cann") + "/set_env.sh"
if not Path(set_env).exists():
    set_env = "/usr/local/Ascend/cann/set_env.sh"

result = subprocess.run(
    ["bash", "-lc", f"source {set_env} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.strip().split("\n"):
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

print("Environment initialization process completed successfully.")


## 2. 什么是核函数

核函数（Kernel Function）是 Ascend C 算子 Device 侧实现的入口函数。

在 Ascend C 中，Host 侧代码负责准备输入输出地址、创建 stream、启动算子并管理同步；Device 侧代码负责在 AI Core 上完成数据搬运、计算和结果写回。核函数就是 Host 侧启动 Device 侧计算时调用的入口。

核函数与普通 C/C++ 函数最大的区别是：普通函数调用通常只在当前执行线程中执行一次；核函数被 Host 侧调用后，会在一个或多个 AI Core 上并行执行。每个参与执行的核都会运行同一份核函数代码，并接收相同的入参。实际处理哪一段数据，需要由核函数内部根据逻辑核 ID 进行区分。


## 3. 核函数的基本定义

SIMD 编程模型下，核函数通常按如下形式定义：


```cpp
__global__ __aicore__ void kernel_name(argument_list);

```

其中包含几个关键规则。

- `__global__`：函数类型限定符，表示该函数是核函数，可以由 Host 侧通过 `<<<...>>>` 内核调用符启动。
- `__aicore__`：函数类型限定符，表示该函数在 Device 侧 AI Core 上执行。
- `void`：核函数必须使用 `void` 返回类型，不能通过返回值把结果传回 Host 侧。
- `argument_list`：核函数入参需要遵循 Ascend C 对函数参数列表的限制。输入、输出和中间配置通常通过指针或标量参数传入。

核函数处理的数据通常存放在 Global Memory（GM）中。对于指向 GM 的指针参数，需要使用变量类型限定符 `__gm__` 标识该指针指向全局内存地址。


```cpp
__global__ __aicore__ void kernel_name(__gm__ uint8_t* input,
                                       __gm__ uint8_t* output);

```

`__gm__ uint8_t*` 是 SIMD 核函数中常见的 GM 指针入参写法。这里统一使用 `uint8_t*` 表达原始字节地址。对于只做打印或控制逻辑的入门示例，核函数也可以没有业务指针入参。


### 3.1 设置核函数的 Kernel 类型

在 SIMD Kernel 定义中，除了 `__global__` 和 `__aicore__`，还可以通过 BuiltIn 关键字显式设置 Kernel 的执行类型。学习者在需要区分 Vector、Cube 或融合执行场景时，可以使用以下方式表达 Kernel 类型，使代码意图更清晰。

- `__vector__`：标识该核函数仅在 Vector 核执行，适用于只包含 Vector 计算或搬运、矢量计算为主的算子，也适合本节 HelloWorld 这类入门 Vector Kernel 示例。
- `__cube__`：标识该核函数仅在 Cube 核执行，适用于矩阵计算等 Cube 计算为主的算子。
- `__mix__(cube, vector)`：标识该核函数同时在 Cube 核和 Vector 核上执行，`cube` 和 `vector` 表示启动时 Cube 核与 Vector 核的配比, 支持的配比为(1, 0)，(0, 1)，(1, 1)， (1, 2)。针对耦合模式的硬件架构，该修饰符不生效。

需要注意：这些关键字主要用于表达 Kernel 在不同计算核上的执行类型。针对耦合模式硬件架构，这类修饰符不生效；在分离模式下，它们可以帮助编译器和运行时更明确地生成、调度对应类型的 Kernel。


```cpp
// Vector Kernel：推荐用于 Vector 计算或入门打印场景。
__global__ __vector__ void hello_world();

// Cube Kernel：推荐用于 Cube 计算为主的算子。
__global__ __cube__ void matmul_custom(__gm__ uint8_t* a,
                                       __gm__ uint8_t* b,
                                       __gm__ uint8_t* c);

// Mix Kernel：用于需要 Cube 与 Vector 协同执行的融合场景。
__global__ __mix__(1, 2) void fused_custom(__gm__ uint8_t* x,
                                           __gm__ uint8_t* y,
                                           __gm__ uint8_t* z);
```


### 3.2 模板核函数

Ascend C 支持使用模板定义核函数。模板参数可以用于在编译期确定数据类型、计算分支或优化策略，使编译器为不同规格生成更专门的代码。常见模板参数包括两类：

- 非类型模板参数：模板参数本身是一个编译期常量，例如 `int repeat`、`uint32_t tileSize`，适合控制循环次数、Tile 大小、流水级数等编译期策略。
- 类型模板参数：模板参数是一个类型，例如 `typename T`，适合为 `half`、`float`、`int32_t` 等不同数据类型生成同一套逻辑的不同实例。

模板核函数的定义形式与普通核函数一致，只是在核函数定义前增加 `template<...>` 参数列表。下面的 `repeat` 是非类型模板参数，用于表达编译期策略；`T` 是类型模板参数，用于约束入参 `value` 的类型：

```cpp
template<int repeat, typename T>
__global__ __vector__ void kernel_name(T value);
```

模板参数在编译期确定，普通函数入参在运行期传入。实际算子中，通常会用模板参数消除运行时分支或适配不同数据类型，再用普通入参传递输入输出地址、长度、Tiling 等运行期数据。


## 4. 核函数调用

本节把核函数调用方式、多核并行语义进行介绍，并对一个核函数如何从 Host 侧下发并在多个逻辑核上执行进行说明。


```cpp
function_name(argument_list);
```


普通 C/C++ 函数调用通常写成上面的形式。核函数需要通过内核调用符 `<<<...>>>` 启动。SIMD 编程模型下，常见调用形式如下：


```cpp
kernel_name<<<numBlocks, dynUBufSize, stream>>>(argument_list);
```


执行配置由 3 个参数决定：

- `numBlocks`：核函数启动的逻辑核数量。每个执行核函数的逻辑核都会获得一个逻辑 ID，可以在核函数中通过内置变量 `block_idx` 获取当前核索引。
- `dynUBufSize`：Dynamic Unified Buffer Size，是配置UB动态内存分配的空间的大小（仅限UB，不包括L1 Buffer等），单位为bytes，默认设置为0。
- `stream`：类型为 `aclrtStream`，用于维护异步操作的执行顺序，确保按照 Host 侧代码下发顺序在 Device 上执行。

针对调用模版核函数，需要在核函数名后提供模板实参，再使用 `<<<numBlocks, dynUBufSize, stream>>>` 配置启动参数。下面的调用假设 Host 侧已经创建了 `aclrtStream stream`，会实例化 `repeat=1`、`T=int` 的核函数版本，并把运行时参数 `7` 传入


```cpp
hello_world<1, int><<<8, 0, stream>>>(7);
```


核函数调用是异步的。Host 侧完成下发后会立即继续执行后续代码，不会默认等待 Device 侧计算结束。如果 Host 侧需要等待该 stream 上的任务完成，可以调用同步接口。


```cpp
aclError ret = aclrtSynchronizeStream(stream);
```


实际工程中，是否需要同步取决于后续逻辑。如果 Host 侧马上要读取 Device 侧输出结果，通常需要同步；如果只是继续向同一个 stream 下发后续 Device 任务，则可以利用 stream 的顺序语义保持异步执行。本节示例显式创建 stream 启动核函数，因此使用 `aclrtSynchronizeStream(stream)` 等待该 stream 上的 Device 侧打印完成。


### 4.1 多核并行执行语义

当核函数以 `numBlocks > 1` 启动时，多个逻辑核会执行同一份核函数代码，并接收同一组入参。HelloWorld 示例中启动参数为 `8`，因此会有 8 个逻辑核执行同一个 `hello_world<1, int>` 核函数，并接收相同的运行时入参 `7`。

在只打印日志的示例中，每个逻辑核可以执行完全相同的逻辑；在处理输入输出数据的真实算子中，为了避免所有核重复处理同一段数据，需要在核函数内部根据逻辑核 ID 划分每个核负责的数据范围。


```cpp
__aicore__ inline void Init(__gm__ uint8_t* input, __gm__ uint8_t* output)
{
    uint32_t blockOffset = block_idx * BLOCK_LENGTH;

    inputGm.SetGlobalBuffer((__gm__ half*)input + blockOffset, BLOCK_LENGTH);
    outputGm.SetGlobalBuffer((__gm__ half*)output + blockOffset, BLOCK_LENGTH);
}

```


以上代码表达的含义是：

- `block_idx` 是内置变量，用于当前核的索引，用于代码内部的多核逻辑控制及多核偏移量计算等。对应的API为`GetBlockIdx`;
- 第 0 个逻辑核处理 `[0, BLOCK_LENGTH)`；
- 第 1 个逻辑核处理 `[BLOCK_LENGTH, 2 * BLOCK_LENGTH)`；
- 第 `block_idx` 个逻辑核处理从 `block_idx * BLOCK_LENGTH` 开始的一段数据。

在真实算子中，还需要结合输入总长度、对齐要求、尾块处理和 Tiling 结果来计算每个核的实际数据范围。这里的示例只用于说明 `block_idx` 在多核数据分片中的作用。


### 4.2 numBlocks 的设置建议

`numBlocks` 是逻辑核数量，取值范围通常为 `[1, 65535]`。为了充分利用硬件资源，Vector Kernel 一般会把 `numBlocks` 设置为可用 AI Core 或 Vector 计算资源数量，或者根据 Tiling 策略设置为其合理倍数。本节 HelloWorld 示例使用 `8`，用于直观看到多个逻辑核分别执行同一核函数。

设置 `numBlocks` 时需要注意：

- `numBlocks` 太小，可能无法充分利用并行计算资源；
- `numBlocks` 太大，可能带来额外调度开销，且不一定提升性能；
- 对数据处理算子来说，每个核处理的数据量需要足够，过细的数据切分可能导致搬运和同步开销占比升高；
- 对仅包含 Vector 计算的 Kernel，`numBlocks` 对应启动的 Vector 逻辑核数量；对仅包含 Cube 计算的 Kernel，`numBlocks` 对应启动的 Cube 逻辑核数量；
- 如果使用了 Device 资源限制能力，`numBlocks` 不应超过运行时可用核数，否则可能破坏资源隔离预期。

实际开发中，`numBlocks` 通常不是随意写死，而是由 Host 侧 Tiling 逻辑结合输入规模、数据类型、对齐要求和平台能力计算得到。


## 5. 三类函数的调用关系

Ascend C 程序中的函数可以分为三类：

- Host 侧执行函数：运行在 CPU 侧，负责资源申请、数据准备、核函数启动和同步等流程。
- 核函数：运行在 Device 侧 AI Core 上，是 Host 侧进入 Device 侧计算逻辑的入口。
- Device 侧普通函数：运行在 Device 侧 AI Core 上，被核函数或其他 Device 侧函数调用，用于拆分和复用计算逻辑。

它们的调用关系可以概括为：

- Host 侧函数可以调用其他 Host 侧函数，也可以通过 `<<<...>>>` 启动核函数；
- 核函数可以调用 Device 侧普通函数；
- Device 侧普通函数可以调用其他 Device 侧普通函数；
- Device 侧代码不能像普通 C++ 调用一样直接调用 Host 侧函数。

![核函数、Host 侧执行函数和 Device 侧普通函数调用关系](./images//03.03.02_simd_ascendc_kernel_function/kernel_function_call_relationship.svg)

因此，复杂算子通常会把 Device 侧逻辑拆成“核函数入口 + 算子类 + 若干成员函数”。


## 6. 核函数开发实践

下面以一个最小可运行的 HelloWorld 算子程序为例说明核函数写法。核函数入口使用 `__global__ __vector__` 声明，表示它由 Host 侧启动，在 Device 侧的 vector 核执行；模板参数 `repeat` 在编译期确定每个逻辑核打印的次数，类型模板参数 `T` 用于演示同一核函数逻辑可以按不同入参类型实例化。

这个示例有三个特点：

- 核函数没有业务输入输出指针，专注演示核函数定义、启动和 Device 侧打印；
- `printf` 来自 `debug/asc_printf.h`，每个被启动的逻辑核都会执行一次打印逻辑；
- `repeat` 是非类型模板参数，调用时通过 `hello_world<1, int>` 给出模板实参；`T` 是类型模板参数，本例实例化为 `int` 并接收运行时入参 `value`。

先执行下面的覆盖写入单元，创建 `Sources/hello_world.asc` 并写入 HelloWorld 示例需要的头文件。


In [ ]:
%%writefile Sources/hello_world.asc

#include "debug/asc_printf.h"
#include "acl/acl.h"


把模板 `hello_world` 核函数 定义加入同一个 `Sources/hello_world.asc` 文件。


In [ ]:
%%writefile -a Sources/hello_world.asc

template<int repeat, typename T>
__global__ __vector__ void hello_world(T value)
{
    for (int i = 0; i < repeat; ++i) {
        printf("Hello World!!! value=%d\n", static_cast<int>(value));
    }
}


### 6.1 Host 侧启动核函数

Host 侧代码负责申请运行资源、创建 stream、启动核函数、等待 Device 侧执行完成、销毁 stream 并释放资源。下面的追加单元会把本节模板 HelloWorld 示例的完整 Host 侧入口写入 `Sources/hello_world.asc`。


Host 侧启动流程与 quickstart 示例保持一致：先调用 `aclrtSetDevice(0)` 申请运行管理资源，然后创建 `aclrtStream stream`，通过三参数内核调用符启动模板核函数，使用 `aclrtSynchronizeStream(stream)` 等待该 stream 上的任务完成，最后销毁 stream 并重置 Device。


In [ ]:
%%writefile -a Sources/hello_world.asc

int main(int argc, char const* argv[])
{
    aclrtSetDevice(0); // 运行管理资源申请。
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);

    // 1. 使用内核调用符<<<numBlock, dynUBufSize, stream>>>调用核函数。
    // 2. numBlock：8表示参与计算的核数为8（核数可根据实际需求设置）。
    // 3. dynUBufSize: 0表示不使用Unified Buffer的动态内存。
    // 4. stream: 类型为aclrtStream，stream用于维护一些异步操作的执行顺序，确保按照应用程序中的代码调用顺序在device上执行。
    hello_world<1, int><<<8, 0, stream>>>(7);
    aclrtSynchronizeStream(stream); // 等待核函数执行完成。
    aclrtDestroyStream(stream); // 销毁流。
    aclrtResetDevice(0); // 销毁运行资源。
    return 0;
}


## 7. CMake 编译配置与架构选择

示例源码生成后，需要通过 CMake 调用毕昇编译器编译 `.asc` 文件。本节默认使用 `dav-3510`，它对应 Ascend 950PR/Ascend 950DT 芯片；`dav-2201` 对应 Atlas A3 训练系列产品、Atlas A3 推理系列产品、Atlas A2 训练系列产品和 Atlas A2 推理系列产品。

下面的 `CMakeLists.txt` 会把 `CMAKE_ASC_ARCHITECTURES` 传给 ASC 编译选项 `--npu-arch`。如果目标设备对应 `dav-2201`，需要同时把默认值和后续 CMake 命令中的 `-DCMAKE_ASC_ARCHITECTURES` 改为 `dav-2201`。


In [ ]:
%%writefile Sources/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU architecture: dav-3510")

find_package(ASC REQUIRED)

project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    hello_world.asc
)
target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)


执行下面的命令配置并编译示例。`build` 目录用于保存 CMake 缓存和编译产物，重新配置编译模式时可以删除该目录后再执行。


In [ ]:
!cmake -S Sources -B Sources/build -DCMAKE_ASC_ARCHITECTURES=dav-3510
!cmake --build Sources/build -j


## 8. 运行 HelloWorld 示例

运行模板 Vector HelloWorld 示例。期望输出中可以看到 8 个逻辑核分别打印 `Hello World!!! value=7`。


In [ ]:
!./Sources/build/demo


## 9. 本节小结

本节介绍了 Ascend C SIMD 编程中的核函数基础，并通过模板 Vector HelloWorld 示例给出了可以编译运行的最小流程。

- 核函数是 Ascend C 程序 Device 侧实现的入口，由 Host 侧通过 `<<<...>>>` 启动；
- Ascend C 支持模板核函数，可以通过类型模板参数和非类型模板参数把数据类型或计算策略放到编译期确定，适合高性能算子中的多规格实现；
- Host 侧函数、核函数和 Device 侧普通函数有明确调用边界，核函数是 Host 进入 Device 侧计算逻辑的入口；
- 一般通过`__vector__`、`__cube__`、`__mix__(cube, vector)`，的关键字设置核函数的Kernel类型；
- `numBlocks` 决定启动的逻辑核数量，核函数内部可以通过内置变量 `block_idx` 获取当前逻辑核 ID；


## 10. 课后练习

本节介绍了 Ascend C SIMD 编程模型中的核函数定义、模板核函数、调用方式和多核并行执行语义，请根据学习内容完成以下题目进行自测。

1. （判断题）SIMD 核函数可以像普通 C++ 函数一样在 Host 侧通过 `kernel(args)` 直接调用执行。

2. （判断题）多个逻辑核执行同一个核函数时，会接收相同的入参，因此通常需要通过内置变量 `block_idx` 区分当前逻辑核处理的数据分片。

3. （单选题）对于一个只需要在vector核上完成Add加法矢量运算的算子，推荐使用哪个 BuiltIn 关键字显式设置 Kernel 类型？  
    A. `__vector__`  
    B. `__cube__`  
    C. `__mix__(1, 2)`  
    D. `__host__`  

4. （单选题）在 SIMD 核函数调用 `kernel_name<<<numBlocks, dynUBufSize, stream>>>(argument_list);` 中，`numBlocks` 表示什么？  
    A. 每个逻辑核内的线程数量  
    B. 核函数启动的逻辑核数量  
    C. 每个核函数动态申请的共享内存大小  
    D. Host 侧输入 Tensor 的数量  

5. （多选题）以下哪些属于编写 SIMD 核函数时常见的注意事项？  
    A. Host 侧通过 `<<<...>>>` 启动核函数  
    B. 核函数调用是异步的，必要时需要通过 stream 或 device 同步等待执行完成  
    C. `numBlocks` 与数据分片逻辑不匹配可能导致数据重复处理或遗漏  
    D. 核函数可以通过返回值把计算结果直接返回给 Host 侧  

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/03.03.02_answer.txt